# Notebook 4 — Sector Clustering
Use unsupervised ML to find clusters of companies with similar financial profiles using K-Means, DBSCAN, and PCA visualisation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120})

DATA = Path('../data/clean')
pl = pd.read_csv(DATA / 'profitandloss.csv')
bs = pd.read_csv(DATA / 'balancesheet.csv')
cf = pd.read_csv(DATA / 'cashflow.csv')
co = pd.read_csv(DATA / 'companies.csv')
an = pd.read_csv(DATA / 'analysis.csv')

pl = pl[pl['is_ttm'] == False]
bs = bs[bs['is_ttm'] == False]
cf = cf[cf['is_ttm'] == False]
print('Data loaded.')

## Step 1 — Feature engineering: one vector per company

In [ ]:
avg_pl = pl.groupby('company_id').agg(
    avg_opm  = ('opm_percentage', 'mean'),
    avg_npm  = ('net_profit_margin_pct', 'mean'),
    avg_div  = ('dividend_payout', 'mean'),
).reset_index()

avg_bs = bs.groupby('company_id').agg(
    avg_de   = ('debt_to_equity', 'mean'),
    avg_eqr  = ('equity_ratio', 'mean'),
).reset_index()

avg_cf = cf.groupby('company_id').agg(
    avg_ocf  = ('operating_activity', 'mean'),
    avg_fcf  = ('free_cash_flow', 'mean'),
).reset_index()

an3 = an[an['period'] == '3Y'][['company_id','compounded_sales_growth_pct']].rename(
    columns={'compounded_sales_growth_pct':'growth_3y'})

roe_df = co[['symbol','roe_percentage']].rename(columns={'symbol':'company_id','roe_percentage':'roe'})

feats = avg_pl.merge(avg_bs, on='company_id', how='outer')
feats = feats.merge(avg_cf, on='company_id', how='outer')
feats = feats.merge(an3, on='company_id', how='left')
feats = feats.merge(roe_df, on='company_id', how='left')
feats = feats.set_index('company_id').fillna(feats.median(numeric_only=True))

scaler = StandardScaler()
X = scaler.fit_transform(feats)
print(f'Feature matrix: {X.shape[0]} companies × {X.shape[1]} features')
print('Features:', list(feats.columns))

## Step 2 — Elbow method: choose best K

In [ ]:
inertias, silhouettes = [], []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(K_range, inertias, marker='o', color='steelblue')
axes[0].set_title('Elbow method — inertia vs K')
axes[0].set_xlabel('Number of clusters K')
axes[0].set_ylabel('Inertia')
axes[1].plot(K_range, silhouettes, marker='o', color='green')
axes[1].set_title('Silhouette score vs K (higher = better)')
axes[1].set_xlabel('Number of clusters K')
axes[1].set_ylabel('Silhouette score')
best_k = K_range[np.argmax(silhouettes)]
axes[1].axvline(best_k, color='red', linestyle='--', alpha=0.7, label=f'Best K={best_k}')
axes[1].legend()
plt.tight_layout()
plt.show()
print(f'Best K by silhouette score: {best_k}')

## Step 3 — K-Means clustering with best K

In [ ]:
km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
feats['kmeans_cluster'] = km.fit_predict(X)

cluster_means = feats.groupby('kmeans_cluster')[list(feats.columns[:-1])].mean().round(2)
print('Cluster profiles:')
print(cluster_means.to_string())

## Step 4 — PCA 2D visualisation

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X)
var_explained = pca.explained_variance_ratio_

plt.figure(figsize=(12, 7))
colors_palette = ['#3b82f6','#10b981','#f59e0b','#ef4444','#8b5cf6','#ec4899']
for k in range(best_k):
    mask = feats['kmeans_cluster'] == k
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1], label=f'Cluster {k}',
                color=colors_palette[k % len(colors_palette)], alpha=0.85, s=80)
    for i, sym in enumerate(feats.index[mask]):
        plt.annotate(sym, (X_2d[mask][i, 0], X_2d[mask][i, 1]),
                     fontsize=6, alpha=0.6, ha='center', va='bottom')
plt.xlabel(f'PC1 ({var_explained[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({var_explained[1]*100:.1f}% variance)')
plt.title('K-Means clusters — PCA 2D projection')
plt.legend()
plt.tight_layout()
plt.show()

## Step 5 — DBSCAN as alternative method

In [ ]:
db = DBSCAN(eps=1.5, min_samples=3)
feats['dbscan_cluster'] = db.fit_predict(X)

print('DBSCAN cluster sizes:')
print(feats['dbscan_cluster'].value_counts())
print(f'Noise points (cluster=-1): {(feats["dbscan_cluster"]==-1).sum()}')

plt.figure(figsize=(12, 7))
for k in sorted(feats['dbscan_cluster'].unique()):
    mask = feats['dbscan_cluster'] == k
    label = f'Noise' if k == -1 else f'Cluster {k}'
    plt.scatter(X_2d[mask.values, 0], X_2d[mask.values, 1],
                label=label, alpha=0.8, s=80)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('DBSCAN clusters — PCA 2D projection')
plt.legend()
plt.tight_layout()
plt.show()

## Step 6 — Label clusters and compare with actual sectors

In [ ]:
CLUSTER_LABELS = {
    0: 'High Growth, Low Debt',
    1: 'Stable Dividend, Moderate Growth',
    2: 'Capital Intensive, High Leverage',
    3: 'Low Margin, High Volume',
    4: 'Premium Profitability',
    5: 'Turnaround / Mixed',
}
feats['cluster_label'] = feats['kmeans_cluster'].map(lambda k: CLUSTER_LABELS.get(k, f'Cluster {k}'))
feats_full = feats.merge(co[['symbol','sector']], left_index=True, right_on='symbol', how='left').set_index('symbol')

fig, ax = plt.subplots(figsize=(12, 6))
cross = pd.crosstab(feats_full['sector'], feats_full['cluster_label'])
cross.plot(kind='bar', ax=ax, alpha=0.85)
ax.set_title('Cluster membership by sector (do IT companies cluster together?)')
ax.set_xlabel('Sector')
ax.set_ylabel('Company count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('\nIT companies cluster assignment:')
print(feats_full[feats_full['sector']=='IT'][['kmeans_cluster','cluster_label']])
print('\nBanking companies cluster assignment:')
print(feats_full[feats_full['sector']=='Banking'][['kmeans_cluster','cluster_label']])